# **National Institute of Justice Recidivism Challenge**
# Assignment Three: Model
*Instructor Copy*

In your groups, use this file to create three machine learning models of your choice based on your cleaned dataset from Assignment 2: The Data. Feel free to refine you data however you need. Each model should be followed by a confusion matrix (include interpretation). Ensure your project is well commented and organized.

<h2>Prepping Work Station</h2>

In [45]:
#Importing libraries
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

#libraries for ML 
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [25]:
#Importing data as a pandas dataframe: df
df = pd.read_csv('cleaned_NIJ_Recidivism_Challenge.csv')

In [26]:
#Viewing first 10 lines
df.head(10)

,ID,Age_at_Release,Residence_PUMA,Gang_Affiliated,Supervision_Risk_Score_First,Supervision_Level_First,Education_Level,Dependents,Prison_Offense,Prison_Years,...,Residence_Changes,Avg_Days_per_DrugTest,DrugTests_THC_Positive,DrugTests_Cocaine_Positive,DrugTests_Meth_Positive,DrugTests_Other_Positive,Percent_Days_Employed,Jobs_Per_Year,Employment_Exempt,Recidivism_Within_3years
0,1,43-47,16,False,3.0,Standard,At least some college,3 or more,Drug,More than 3 years,...,2,612.000000,0.000000,0.000000,0.000000,0.00000,0.488562,0.447610,False,False
1,2,33-37,16,False,6.0,Specialized,Less than HS diploma,1,Violent/Non-Sex,More than 3 years,...,2,35.666667,0.000000,0.000000,0.000000,0.00000,0.425234,2.000000,False,True
2,3,48 or older,24,False,7.0,High,At least some college,3 or more,Drug,1-2 years,...,0,93.666667,0.333333,0.000000,0.166667,0.00000,0.000000,0.000000,False,True
3,4,38-42,16,False,7.0,High,Less than HS diploma,1,Property,1-2 years,...,3 or more,25.400000,0.000000,0.000000,0.000000,0.00000,1.000000,0.718996,False,False
4,5,33-37,16,False,4.0,Specialized,Less than HS diploma,3 or more,Violent/Non-Sex,1-2 years,...,0,23.117647,0.000000,0.000000,0.058824,0.00000,0.203562,0.929389,False,True
5,6,38-42,17,False,5.0,Standard,High School Diploma,0,Property,More than 3 years,...,3 or more,474.600000,0.000000,0.000000,0.000000,0.00000,0.674252,0.307838,False,False
6,7,48 or older,18,False,2.0,Standard,Less than HS diploma,2,Property,Less than 1 year,...,1,238.500000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,False,True
7,8,38-42,16,False,5.0,High,High School Diploma,3 or more,Drug,Greater than 2 to 3 years,...,0,34.775000,0.000000,0.000000,0.000000,0.00000,0.469446,0.525162,False,False
8,9,43-47,5,False,7.0,High,High School Diploma,0,Drug,Less than 1 year,...,0,93.890044,0.063350,0.013741,0.012890,0.00755,0.000000,0.000000,True,True
9,10,43-47,16,False,5.0,Standard,High School Diploma,3 or more,Property,More than 3 years,...,0,27.800000,0.000000,0.000000,0.000000,0.00000,0.359712,1.000000,False,True


<h5>Some final cleaning to prepare it for sklearn</h5>

In [27]:
def convert_to_number(value):
    """Convert strings with 'or more', 'More than', etc. to a number by removing the text."""
    if isinstance(value, str):
        value = value.lower()
        value = re.sub(r'[^\d.]', '', value)  # Remove non-numeric characters
        try:
            return float(value)
        except ValueError:
            return np.nan  # Return NaN if conversion fails
    try:
        return float(value)
    except ValueError:
        return np.nan  # Return NaN if conversion fails

def preprocess_dataset(df):
    """Preprocess the dataset by converting appropriate columns to numerical values and encoding categorical columns."""
    # Convert 'Age_at_Release' column specifically
    def convert_age(age):
        if isinstance(age, str):
            if '-' in age:
                return float(sum(map(int, age.split('-'))) / 2)
            elif age == '48 or older':
                return 48.0
            else:
                return float(re.sub(r'[^\d.]', '', age))
        else:
            return age 
    df['Age_at_Release'] = df['Age_at_Release'].apply(convert_age)

    # Columns to be converted using convert_to_number function
    columns_to_convert = [
        'Dependents', 'Prison_Years', 'Prior_Arrest_Episodes_Felony',
        'Prior_Arrest_Episodes_Misd', 'Prior_Arrest_Episodes_Violent',
        'Prior_Arrest_Episodes_Property', 'Prior_Arrest_Episodes_Drug',
        'Prior_Arrest_Episodes_PPViolationCharges', 'Prior_Conviction_Episodes_Felony',
        'Prior_Conviction_Episodes_Misd', 'Prior_Conviction_Episodes_Prop',
        'Prior_Conviction_Episodes_Drug', 'Delinquency_Reports',
        'Program_Attendances', 'Program_UnexcusedAbsences', 'Residence_Changes'
    ]

    for col in columns_to_convert:
        if col in df.columns:
            df[col] = df[col].apply(convert_to_number)

    # Convert boolean columns to integers
    boolean_columns = df.select_dtypes(include='bool').columns
    df[boolean_columns] = df[boolean_columns].astype(int)

    # Convert categorical columns using one-hot encoding
    categorical_columns = [
        'Supervision_Level_First', 'Education_Level', 'Prison_Offense'
    ]

    df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

    # Fill any NaN values that resulted from conversion issues
    df.fillna(0, inplace=True)

    return df

In [28]:
df = preprocess_dataset(df)

<h2>Splitting the Data</h2>

In [29]:
#Defining Variables
    # X contains features
    # y contains target variable

X = df.drop(columns=['Recidivism_Within_3years', 'ID'])
y = df['Recidivism_Within_3years']

#Splitting data into (70%) training and (30%) test sets 
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state = 42)

#Checking shapes of sets
print("X Training: " + str(X_train.shape))
print("y Training: " + str(y_train.shape))
print ("X Test: " + str(X_test.shape))
print("y Test: " + str(y_test))

X Training: (18084, 51)
y Training: (18084,)
X Test: (7751, 51)
y Test: 10906    1
10658    1
9500     0
16797    1
10324    1
        ..
17688    0
19046    1
7568     1
13825    1
24003    1
Name: Recidivism_Within_3years, Length: 7751, dtype: int64


In [30]:
# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

<h2>Model One</h2>
Logistic Regression

In [31]:
#Initializing model
logistic = LogisticRegression(C = 1.0) #Hyperparamter -> regularization paramter C = 1.0
#Fitting model to data
logistic.fit(X_train_scaled, y_train)
#Implementign model
logistic_predictions = logistic.predict(X_test_scaled)

In [32]:
#Confusion Matrix
print(confusion_matrix(y_test, logistic_predictions))

[[1999 1323]
 [ 889 3540]]


The Logistic Regression Confusion Matrix:

    True Negatives: 1999
    False Positive: 1323
    False Negative: 889
    True Positive: 3540

**Accuracy**

The proportion of correct predictions

(TP + TN)/(TP + TN + FP + FN)

In [33]:
LRaccuracy = ((3540 + 1999)/(3540 + 1999 + 1323 + 889)) * 100 
print(f'{LRaccuracy}%')

71.46174687137143%


**Precision**  *Positive Predictive Value*

The proportion of correct positive predictions

TP/(TP + FP)

In [34]:
LRpercision = (3540/(3540 + 1323)) * 100
print(f'{LRpercision}%')

72.79457125231339%


**Recall** *True Positive Rate*

Proportion of positives correctly identified

TP/(TP + FN)

In [35]:
LRrecall = (3540/(3540 + 889)) * 100
print(f'{LRrecall}%')

79.92774892752314%


**Specificity** *True Negative Rate*

Proportion of negatives correctly identified

TN/(TN + FP)

In [36]:
LRspecificity = (1999/(1999 + 1323)) * 100
print(f'{LRspecificity}%')

60.17459361830223%


**F1 Score**

The mean of precision and recall

2 * ((Precision * Recall)/(Precision + Recall))

In [37]:
LRf1 = 2  * ((LRpercision * LRrecall)/(LRpercision + LRrecall))
print(LRf1)

76.19457597933707


<h5>Interpretation</h5>

Accuracy = 71.5%

    The logistic regression model is correct 71.5% of the time

Precision = 72.8%

    When the model said a prisoner recidivated, it was correct 72.8% of the time.

Recall = 79.9%

    The model correctly predicted yes for 79.9% of all recividated prisoners.

Specificity = 60.2%

    The model correctly predicted no for 60.2% of prisoners that did not return to prison



F1 = 76.2%

    The model preformed well, but there is still serious room for improvement especially in predicting negative cases

<h2>Model Two</h2>
K-NN Imputation

In [38]:
#Creating KNN Model
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)
y_predictions = knn.predict(X_test)

In [39]:
#Printing confusion matrix
print(confusion_matrix(y_test, y_predictions))

[[1615 1707]
 [1201 3228]]


The KNN Imputation Confusion Matrix:

    True Negatives: 1615
    False Positive: 1707
    False Negative: 1201
    True Positive: 3228

**Accuracy**

The proportion of correct predictions

(TP + TN)/(TP + TN + FP + FN)

In [40]:
KNNaccuracy  = ((3228 + 1615)/(3228 + 1615 + 1707 + 1201)) * 100
print(f'{KNNaccuracy}%')

62.482260353502774%


**Precision**  *Positive Predictive Value*

The proportion of correct positive predictions

TP/(TP + FP)

In [41]:
KNNprecision = (3228/(3228 + 1707)) * 100
print(f'{KNNprecision}%')

65.41033434650456%


**Recall** *True Positive Rate*

Proportion of positives correctly identified

TP/(TP + FN)

In [42]:
KNNrecall = (3228/(3228 + 1201)) * 100
print(f'{KNNrecall}%')

72.88326936102958%


**Specificity** *True Negative Rate*

Proportion of negatives correctly identified

TN/(TN + FP)

In [43]:
KNNspecificity = (1615/(1615 + 1707)) * 100
print(f'{KNNspecificity}%')

48.61529199277544%


**F1 Score**

The mean of precision and recall

2 * ((Precision * Recall)/(Precision + Recall))

In [44]:
KNNf1 = 2 * ((KNNprecision * KNNrecall)/(KNNprecision + KNNrecall))
print(f'{KNNf1}%')

68.94489534387013%


<h5>Interpretation</h5>

Accuracy = 62.5%

    The KNN Imputation Model only made the correct choice 62.5% of the time

Precision = 65.4%

    The model was correct in its positive predictions 65.4% of the time    

Recall = 72.8%

    The model correctly predicted yes for 72.8% of all recividated prisoners.

Specificity = 48.6%

    The model correctly predicted no for 48.6% of all prisoners that did not return to prison.    



F1 = 68.9%

    Our model - while performing at a level better than guessing - does not contain any actual preditive value
   

<h2>Model Three</h2>
Random Forest

In [46]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [48]:
print(confusion_matrix(y_test, y_pred))

[[1974 1348]
 [ 763 3666]]


The Random Forest Confusion Matrix:

    True Negatives: 1974
    False Positive: 1348
    False Negative: 763
    True Positive: 3666

**Accuracy**

The proportion of correct predictions

(TP + TN)/(TP + TN + FP + FN)

In [1]:
RFaccuracy = ((3666 + 1974)/(3666 + 1974 + 1348 + 763)) * 100
print(f'{RFaccuracy}%')

72.7648045413495


**Precision**  *Positive Predictive Value*

The proportion of correct positive predictions

TP/(TP + FP)

In [2]:
RFprecision = (3666/(3666 + 1348)) * 100
print(f'{RFprecision}%')

73.11527722377343


**Recall** *True Positive Rate*

Proportion of positives correctly identified

TP/(TP + FN)

In [3]:
RFrecall = (3666/(3666 + 763)) * 100
print(f'{RFrecall}%')

82.77263490629939%


**Specificity** *True Negative Rate*

Proportion of negatives correctly identified

TN/(TN + FP)

In [4]:
RFspecificity = (1974/(1974 + 1348)) * 100
print(f'{RFspecificity}%')

59.42203491872367%


**F1 Score**

The mean of precision and recall

2 * ((Precision * Recall)/(Precision + Recall))

In [5]:
RFf1 = 2 * ((RFprecision * RFrecall)/(RFprecision + RFrecall))
print(f'{RFf1}%')

77.64481626601714%


<h5>Interpretation</h5>

Accuracy = 72.8%

    The Random Forest Model made a correct prediction 72.8% of the time

Precision = 73.1%

    The model was correct in its positive predictions 73.1% of the time    

Recall = 82.8%

    The model correctly predicted yes for 2.8% of all recividated prisoners.

Specificity = 59.4%

    The model correctly predicted no for 59.4% of all prisoners that did not return to prison.    



F1 = 77.6%

    This model is our best preforming one, however there is still room for improvement as a model is not considered "good" until it reaches an 80% F1 score.
   